# 🔬 BioSLATE Project — Phase 2 & 3: Synthetic Lethal Target Discovery

**Author:** Faith Ogundimu  
**Date:** June 2025

**Notebook Purpose:**
This notebook sets the foundation for identifying synthetic lethal interactions in High-Grade Serous Ovarian Cancer (HGSOC) cell lines. Using gene effect scores from DepMap CRISPR screens (Chronos), we aim to determine which genes become essential when specific genomic biomarkers (e.g., BRCA1 deletion) are present. These biomarkers were defined through cross-validation between TCGA tumours and DepMap cell lines (Phase 1).

> **Context:** This analysis builds directly on the biomarker shortlists from Phase 1, where amplified and deleted genes were cross-validated between TCGA tumours and HGSOC cell lines.
> Here, we test whether these CNA biomarkers can stratify cell lines in a way that reveals context-specific gene dependencies.

### 🎯 Objectives:
* Group HGSOC cell lines based on the presence or absence of validated CNA biomarkers (e.g., BRCA1 deletion).
* For each group, compare CRISPR gene effect scores to identify genes that are **synthetically lethal** only in biomarker-positive lines.
* Perform statistical tests (e.g., Welch’s t-test) to assess significance.
* Adjust for multiple testing using False Discovery Rate (FDR) correction.
* Prioritise biomarker–target pairs with strong effect sizes and high confidence for therapeutic relevance.


In [11]:
import pandas as pd

# read in file
crispr = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/database_files/DepMap/CRISPRGeneEffect.csv", index_col=0)
# This will make 'ACH-000628', 'ACH-000974', etc., become the row index (instead of just Row 0, Row 1, ...).

In [2]:
crispr

,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),AADAC (13),...,ZWILCH (55055),ZWINT (11130),ZXDA (7789),ZXDB (158586),ZXDC (79364),ZYG11A (440590),ZYG11B (79699),ZYX (7791),ZZEF1 (23140),ZZZ3 (26009)
ACH-000001,-0.121964,0.042647,0.026102,-0.147614,-0.020425,0.184014,-0.076020,-0.097068,0.182759,-0.002365,...,-0.120536,0.107286,0.166390,0.262275,0.163876,-0.249508,0.053725,0.067986,0.136285,-0.438536
ACH-000004,-0.016484,-0.088500,-0.088198,-0.041225,-0.028074,-0.157206,0.280101,-0.106904,0.178125,0.149760,...,-0.192527,-0.324059,0.230377,0.087609,0.074897,0.054335,-0.330343,0.099067,0.274566,0.001871
ACH-000005,-0.184847,0.003300,0.160881,0.086224,-0.149315,-0.253837,0.167011,-0.101209,-0.129827,0.029143,...,-0.312827,-0.338023,-0.039700,-0.055349,-0.000367,-0.205605,-0.066032,-0.054518,0.035579,-0.150486
ACH-000007,-0.071921,-0.113717,0.082872,0.099633,-0.008378,-0.022310,0.014416,-0.184977,-0.173739,0.196363,...,-0.334843,-0.355499,-0.014183,0.230944,0.044628,-0.081863,-0.390748,-0.036547,-0.273129,-0.382723
ACH-000009,-0.019163,-0.134669,0.060323,0.076647,0.078922,-0.100243,0.047559,-0.136988,0.037759,0.093207,...,-0.299593,-0.194427,-0.027365,0.236591,-0.084224,-0.098271,-0.510495,0.052938,0.018623,-0.258353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-002926,-0.107883,-0.101501,0.046677,0.096868,-0.118650,-0.065258,-0.064944,-0.175912,-0.044707,0.126244,...,-0.409541,-0.579376,0.038706,0.338610,-0.174379,-0.040264,-0.239763,0.065168,-0.144088,-0.334424
ACH-002928,-0.287606,-0.179101,-0.135824,-0.046579,-0.249888,-0.097376,0.070385,0.049444,-0.244937,0.058041,...,-0.095328,-0.440452,0.303500,0.097690,0.151790,0.039994,0.098240,-0.266654,-0.115108,-0.080421
ACH-003012,-0.145878,-0.004000,0.091933,0.191332,-0.065385,-0.134152,0.012353,-0.321012,0.001571,0.146177,...,-0.280119,-0.662673,-0.180936,0.169997,-0.036326,0.158702,-0.213281,-0.017884,-0.195395,-0.309021
ACH-003177,-0.038435,-0.055660,0.018989,0.117182,0.063397,0.049993,0.175520,-0.085752,0.078435,0.089513,...,-0.080700,-0.476797,-0.158375,-0.008999,-0.030058,-0.103513,-0.119234,-0.153744,0.032584,-0.536071


### 🧬 What is the "CRISPR Gene Effect" (Chronos)?

- Each **value** reflects how essential a gene is for a given **cell line's survival**.
- **Lower scores** (closer to –1) = **higher dependency** (i.e. gene knockout causes cell death).
- **Score ~ 0** = knockout has little to no effect.
- Chronos is an updated model designed to better estimate **gene knockout effects** from CRISPR screens, especially in **non-dividing or slow-growing** cells.

| **Gene Effect Score** | **Interpretation** |
| --- | --- |
| ~–1.0 | Strong dependency |
| ~0.0 | No effect |
| > 0.0 | Possibly beneficial knock-out (rare) |

In [3]:
hgsoc_cell_lines = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/database_files/DepMap/cell lines in High-Grade Serous Ovarian Cancer.csv")
# The first row is treated as the header (column names) by default. (explicit; header=0)

hgsoc_cell_lines

,Depmap Id,Cell Line,Primary Disease,Tumor Type
0,ACH-000132,JHOS2,Ovarian Epithelial Tumor,Primary
1,ACH-002183,OVMIU,Ovarian Epithelial Tumor,NaN
2,ACH-001151,OVCAR5,Ovarian Epithelial Tumor,Metastatic
3,ACH-000635,SNU119,Ovarian Epithelial Tumor,Metastatic
4,ACH-000430,TYKNU,Ovarian Epithelial Tumor,Primary
5,ACH-002140,HEY,Ovarian Epithelial Tumor,Primary
6,ACH-000103,CAOV4,Ovarian Epithelial Tumor,Metastatic
7,ACH-000001,NIHOVCAR3,Ovarian Epithelial Tumor,Metastatic
8,ACH-000256,COV318,Ovarian Epithelial Tumor,Metastatic
9,ACH-000542,HEYA8,Ovarian Epithelial Tumor,Metastatic


In [9]:
# Get the intersection of cell lines present in both hgsoc_cell_lines and crispr
valid_ids = hgsoc_cell_lines["Depmap Id"].isin(crispr.index) # returns boolean
shared_ids = hgsoc_cell_lines.loc[valid_ids, "Depmap Id"] # returns indexed list.

# Now use this list to filter crispr
crispr_filtered = crispr.loc[shared_ids]

In [10]:
crispr_filtered.columns = [col.split(" ")[0] for col in crispr_filtered.columns]
# ['ACH-000635', 'ACH-000409', 'ACH-000574', 'ACH-000617', 'ACH-000443'] not in crispr file.

### CRISPR File 

In [47]:
crispr_filtered

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000132,-0.122873,0.072059,0.038849,0.039049,-0.209066,0.076330,-0.094047,-0.009645,-0.031795,0.202345,...,-0.131236,-0.471731,0.029672,0.012413,-0.127275,0.018842,-0.164656,-0.155827,-0.115392,-0.650192
ACH-002183,-0.245112,-0.053354,0.071037,0.337962,-0.109942,-0.013398,0.303089,0.046424,0.200121,0.492939,...,-0.405278,-0.548039,0.075445,-0.046378,-0.319488,-0.146330,-0.352998,-0.081459,-0.191924,-0.215067
ACH-001151,-0.102453,-0.077385,-0.019409,0.028298,-0.160307,-0.056900,0.039573,-0.268167,0.002770,0.096360,...,-0.119769,-0.217415,0.016761,0.170917,-0.044003,-0.095041,-0.187164,-0.083624,-0.176731,-0.417039
ACH-000430,-0.084244,0.058017,-0.053142,0.198323,-0.072941,-0.206886,0.122309,-0.018023,-0.105048,0.214323,...,-0.023189,-0.538709,0.027976,0.119626,-0.031476,-0.021702,-0.215309,-0.333313,-0.015074,-0.006619
ACH-002140,-0.104728,0.110308,-0.025987,0.257998,0.108899,-0.154841,0.001961,-0.214199,0.055091,0.245402,...,-0.665836,-0.157587,0.073392,-0.048359,-0.063494,-0.012150,-0.228079,-0.016597,-0.187425,-0.560568
ACH-000103,-0.294804,0.065371,-0.063194,0.072800,-0.072182,0.159779,0.236855,0.137420,-0.244539,0.331102,...,-0.269222,-0.593750,0.108921,-0.102701,-0.173919,0.045712,-0.208987,0.035390,-0.158814,0.033504
ACH-000001,-0.121964,0.042647,0.026102,-0.147614,-0.020425,0.184014,-0.076020,-0.097068,0.182759,-0.002365,...,-0.120536,0.107286,0.166390,0.262275,0.163876,-0.249508,0.053725,0.067986,0.136285,-0.438536
ACH-000256,-0.198217,-0.046660,0.050505,0.029620,-0.062260,-0.002114,0.278037,-0.293802,-0.109867,0.157249,...,-0.304890,-0.471835,-0.124134,0.080740,0.161809,0.089320,0.025617,-0.091058,0.117985,-0.217564
ACH-000542,-0.135195,-0.012998,0.106194,0.150954,0.033209,-0.006644,-0.015868,-0.542180,-0.093912,0.089950,...,-0.321037,-0.365938,0.045471,0.107203,-0.067005,0.014995,-0.359367,-0.094584,-0.216769,-0.449894
ACH-000696,-0.014307,-0.008415,0.023248,0.063404,-0.050467,-0.005701,-0.025869,-0.297252,-0.071535,0.081685,...,-0.179381,-0.542944,0.105737,-0.027614,-0.185295,-0.033858,-0.113058,-0.103185,-0.238914,-0.439584


### Group HGSOC cell lines based on the presence or absence of validated CNA biomarkers (e.g., BRCA1 deletion)

In [37]:
# Read in candidate deleted biomarkers.
cna_hgsoc_depmap = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/results/log2_cna_depmap_hgsoc.csv", index_col=0)

In [25]:
cna_hgsoc_depmap

,RHEB,TIPIN,OR4A47,NUP133,CPNE8-AS1,PDIA4,RNU6-899P,NRXN1-DT,SMIM41,RPS12P2,...,RN7SKP98,RPSAP15,FAM149B1P1,RNU6-1020P,MIR1285-1,FXYD5,RNA5SP154,TMEM87B,RNU6-1313P,RNU6-1158P
ACH-000132,1.029368,0.675483,1.149846,0.867362,1.044429,1.029368,1.141747,1.114393,0.994776,1.102393,...,0.479031,1.430699,1.113241,1.104895,0.661686,1.119210,0.705264,0.935732,1.195824,1.195824
ACH-002183,0.981266,0.951344,0.964016,1.336421,0.981230,0.981266,0.964016,0.967135,0.981230,0.983182,...,1.301606,0.852798,0.951344,0.974614,0.993510,1.000375,0.974561,0.977713,0.981904,0.981904
ACH-001151,0.921363,1.174023,1.197960,0.884346,0.959600,0.921363,1.190102,1.258202,0.858686,0.925456,...,0.937907,0.577166,1.229583,0.865415,0.935343,1.063702,0.962042,1.185818,1.144143,1.463407
ACH-000635,1.262903,0.753092,1.077517,1.291315,1.307322,1.262903,1.081227,1.072795,1.022470,0.790750,...,1.080605,0.474356,0.789536,0.750697,1.300275,1.262426,0.833488,1.308682,1.031222,1.063975
ACH-000430,1.420414,0.884105,1.156367,1.168365,0.890775,1.420414,1.156367,1.167760,0.890775,0.879134,...,1.168365,0.804875,0.874417,0.880093,1.414913,0.900395,0.867872,1.148953,0.914548,0.914548
ACH-002140,0.338121,0.873629,1.011571,0.869052,0.948762,0.338121,1.011571,1.011472,0.948762,1.027690,...,0.828571,0.561849,0.873629,1.150473,0.623486,1.040164,0.842503,1.017212,1.172833,1.172833
ACH-000103,1.165262,0.850423,0.808155,1.455622,1.146830,1.165262,0.808155,1.131607,1.146830,0.869160,...,0.966479,0.459685,0.841847,0.869173,1.165262,1.402803,1.175063,1.131607,1.153039,1.153039
ACH-000001,0.956619,1.056391,1.229718,1.119576,1.421059,0.956749,1.155812,0.959903,1.154655,1.162089,...,1.098675,0.978328,0.698191,1.225705,0.964623,1.481259,1.415230,0.960568,1.204170,1.466337
ACH-000256,1.392006,0.855593,0.691193,1.317037,1.125519,1.392006,1.130639,1.197064,0.891266,0.933399,...,1.330821,0.731294,0.880376,0.887462,1.451515,1.575292,0.899918,1.148287,0.897523,0.897523
ACH-000542,0.620004,0.835363,1.048825,0.831059,1.087975,0.620004,1.027667,1.038759,1.023777,1.066779,...,0.877901,0.683157,0.835363,1.158517,0.647264,0.998326,0.908402,1.041962,1.111500,1.115853


In [35]:
# Read in candidate deleted biomarkers.
del_biomarkers = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/results/cross_val_del_sig_genes.csv", index_col=1)

In [36]:
del_biomarkers = del_biomarkers.index
del_biomarkers

Index(['COPS2', 'CTDP1', 'CTPS2', 'CTSB', 'CWC27', 'CYB5A', 'CYB5B', 'CYB5R3',
       'DAPK2', 'DAZAP1',
       ...
       'VPS39', 'WDR7', 'WTAP', 'WWOX', 'XPO7', 'XRCC4', 'YTHDC2', 'ZC3H18',
       'ZFYVE16', 'ZFYVE19'],
      dtype='object', name='Gene', length=155)

In [38]:
cna_hgsoc_depmap = cna_hgsoc_depmap.transpose()
cna_hgsoc_depmap_filtered = cna_hgsoc_depmap.loc[del_biomarkers]

In [41]:
cna_hgsoc_depmap_filtered
# Genes are index

,ACH-000132,ACH-002183,ACH-001151,ACH-000635,ACH-000430,ACH-002140,ACH-000103,ACH-000001,ACH-000256,ACH-000542,...,ACH-000524,ACH-000617,ACH-000116,ACH-000520,ACH-000278,ACH-000584,ACH-000713,ACH-000013,ACH-001628,ACH-000443
Gene,,,,,,,,,,,,,,,,,,,,,
COPS2,0.675483,0.951344,1.225255,0.444341,0.884105,0.873629,0.850423,1.084235,0.865046,0.835363,...,0.822783,0.780983,0.852949,0.497701,1.100746,0.992830,0.757106,1.021346,0.815331,0.990342
CTDP1,0.667126,0.967067,0.551166,0.783474,0.866100,0.610792,0.841677,0.695114,0.905888,0.644297,...,0.507102,0.460995,1.152507,0.517161,0.690865,1.488787,0.681661,0.773340,0.984017,1.037108
CTPS2,0.984852,0.852798,0.554079,0.469790,0.487987,0.516167,0.459685,0.713244,0.415040,0.545054,...,0.519930,0.806326,0.640523,0.897367,0.712337,1.109468,0.741549,1.358492,0.546303,0.517654
CTSB,0.700382,0.561640,0.549510,0.788529,1.152813,0.746203,0.786048,0.914799,0.913409,0.871418,...,0.498876,0.797526,0.830151,0.880383,0.673170,0.428082,1.026240,0.766686,1.134842,0.899725
CWC27,0.940117,0.950502,1.507675,0.794723,1.164499,1.013792,1.081030,1.003044,0.946026,1.076135,...,0.740611,0.797398,0.593975,1.168819,0.678559,0.754565,0.745935,1.080524,0.589877,0.937193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XRCC4,1.146639,0.948809,0.917064,0.789331,1.164499,1.013792,1.081030,0.947546,0.695273,1.076135,...,0.740611,0.797398,0.967410,1.168819,0.678559,0.754565,1.329787,1.024452,0.585307,0.937193
YTHDC2,0.778609,0.962731,1.483577,0.789331,1.155586,1.013792,1.081030,0.971854,1.095151,1.076135,...,0.740611,1.305877,1.021215,1.176070,0.690802,0.748079,1.298424,1.074694,0.585307,0.937193
ZC3H18,1.115957,1.002597,0.964551,0.794233,1.178516,0.989632,0.939017,0.703878,1.079186,0.979826,...,0.877416,0.795669,0.970855,0.498568,0.704466,1.351828,0.642022,0.835498,1.047285,0.702011


### CNA File

In [44]:
cna_hgsoc_depmap_filtered = cna_hgsoc_depmap_filtered.transpose()
cna_hgsoc_depmap_filtered

Gene,COPS2,CTDP1,CTPS2,CTSB,CWC27,CYB5A,CYB5B,CYB5R3,DAPK2,DAZAP1,...,VPS39,WDR7,WTAP,WWOX,XPO7,XRCC4,YTHDC2,ZC3H18,ZFYVE16,ZFYVE19
ACH-000132,0.675483,0.667126,0.984852,0.700382,0.940117,0.667126,1.050673,0.867376,0.675483,0.804100,...,0.675483,0.667126,0.819540,0.837246,0.705744,1.146639,0.778609,1.115957,1.088187,0.675483
ACH-002183,0.951344,0.967067,0.852798,0.561640,0.950502,0.967067,1.002597,0.990702,0.951344,1.311147,...,0.951344,0.967067,1.034575,1.002597,0.561640,0.948809,0.962731,1.002597,1.324414,0.951344
ACH-001151,1.225255,0.551166,0.554079,0.549510,1.507675,0.551166,0.964551,0.876535,1.174023,1.047634,...,1.225255,0.551166,0.930106,0.964551,0.532021,0.917064,1.483577,0.964551,0.917064,1.225255
ACH-000635,0.444341,0.783474,0.469790,0.788529,0.794723,0.783474,0.805575,0.757142,0.753092,0.744968,...,0.751390,0.826253,0.786928,1.107235,0.797684,0.789331,0.789331,0.794233,0.789331,1.072092
ACH-000430,0.884105,0.866100,0.487987,1.152813,1.164499,0.866100,1.178516,0.886258,0.884105,0.901020,...,0.885953,0.866100,0.488067,1.178516,1.152813,1.164499,1.155586,1.178516,1.164499,0.885953
ACH-002140,0.873629,0.610792,0.516167,0.746203,1.013792,0.610792,0.989632,1.024539,0.873629,0.852519,...,0.873629,0.610792,1.105455,0.989632,0.746203,1.013792,1.013792,0.989632,1.013792,0.873629
ACH-000103,0.850423,0.841677,0.459685,0.786048,1.081030,0.877338,0.939017,0.865709,0.850423,0.915016,...,0.850423,0.877338,0.590004,0.939017,0.786048,1.081030,1.081030,0.939017,1.081030,0.861858
ACH-000001,1.084235,0.695114,0.713244,0.914799,1.003044,0.710520,0.697585,0.680799,1.056391,0.685484,...,1.047450,1.206826,0.700798,0.711740,1.109463,0.947546,0.971854,0.703878,0.947546,1.047450
ACH-000256,0.865046,0.905888,0.415040,0.913409,0.946026,0.905888,0.907422,0.861073,0.855593,0.834776,...,0.831461,0.905888,0.898935,0.906928,0.906554,0.695273,1.095151,1.079186,0.695273,0.891083
ACH-000542,0.835363,0.644297,0.545054,0.871418,1.076135,0.644297,0.979826,0.922265,0.835363,1.006786,...,0.835363,0.644297,1.054981,0.979826,0.859626,1.076135,1.076135,0.979826,1.076135,0.835363


In [49]:
# 155 biomarkers × 18,000 genes = 2.79 million biomarker–target comparisons

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# Parameters
DELETION_THRESHOLD = 0.9  # log2(CN + 1)

# --- Step 1: Generate CNA status matrix for all deleted biomarkers ---
status_dict = {}

for gene in del_biomarkers:
    if gene in cna_hgsoc_depmap_filtered.columns:
        status_dict[f"{gene}_status"] = cna_hgsoc_depmap_filtered[gene].apply(
            lambda x: "Deleted" if x <= DELETION_THRESHOLD else "WT"
        )

# Create single CNA status dataframe in one operation (improves memory performance)
cna_status_df = pd.DataFrame(status_dict, index=cna_hgsoc_depmap_filtered.index).copy()

# --- Step 2: Loop through biomarkers and perform CRISPR comparison ---
results = []

for biomarker in del_biomarkers:
    biomarker_col = f"{biomarker}_status"
    
    # Skip genes not present in the CNA status matrix
    if biomarker_col not in cna_status_df.columns:
        continue

    # Get biomarker-positive and WT groups
    deleted_lines = cna_status_df[cna_status_df[biomarker_col] == "Deleted"].index
    wt_lines = cna_status_df[cna_status_df[biomarker_col] == "WT"].index

    # Filter to only lines in CRISPR dataset
    deleted_lines = [line for line in deleted_lines if line in crispr_filtered.index]
    wt_lines = [line for line in wt_lines if line in crispr_filtered.index]

    # Skip small groups
    if len(deleted_lines) < 3 or len(wt_lines) < 3:
        continue

    # --- Step 3: Loop through each gene in CRISPR data ---
    for gene in crispr_filtered.columns:
        group_del = crispr_filtered.loc[deleted_lines, gene].dropna()
        group_wt = crispr_filtered.loc[wt_lines, gene].dropna()

        if len(group_del) < 3 or len(group_wt) < 3:
            continue

        # Welch's t-test
        t_stat, p_val = ttest_ind(group_del, group_wt, equal_var=False)

        # Cohen's d effect size
        pooled_sd = np.sqrt(((group_del.std(ddof=1)**2 + group_wt.std(ddof=1)**2) / 2))
        effect_size = (group_del.mean() - group_wt.mean()) / pooled_sd if pooled_sd > 0 else np.nan

        results.append({
            "Biomarker": biomarker,
            "TargetGene": gene,
            "T-stat": t_stat,
            "P-value": p_val,
            "EffectSize": effect_size,
            "n_Deleted": len(group_del),
            "n_WT": len(group_wt),
            "MeanEffect_Deleted": group_del.mean(),
            "MeanEffect_WT": group_wt.mean()
        })

# --- Step 4: Convert to DataFrame ---
results_df = pd.DataFrame(results)

# --- Step 5: Adjust p-values with FDR correction ---
results_df["FDR"] = results_df.groupby("Biomarker")["P-value"].transform(
    lambda p: multipletests(p, method="fdr_bh")[1]
)

# --- Step 6: Save to file ---
results_df.to_csv("../results/synthetic_lethality_screen.csv", index=False)

In [51]:
import pandas as pd

# Load results
results_df = pd.read_csv("../results/synthetic_lethality_screen.csv")

# Filter 1: P-value < 0.05
pval_hits = results_df[results_df["P-value"] < 0.05]
pval_hits.to_csv("../results/significant_synthetic_hits_pval.csv", index=False)

# Filter 2: FDR < 0.05
fdr_hits = results_df[results_df["FDR"] < 0.05]
fdr_hits.to_csv("../results/significant_synthetic_hits_fdr.csv", index=False)

# Filter 3: Druggable synthetic lethals
# Criteria: Statistically significant (FDR < 0.05) and negative effect size
# Negative effect size = more essential in biomarker-deleted cell lines
druggable_hits = results_df[
    (results_df["FDR"] < 0.05) &
    (results_df["EffectSize"] < 0)
]
druggable_hits.to_csv("../results/druggable_synthetic_lethal_hits.csv", index=False)

# Print summary
print(f"P-value < 0.05: {len(pval_hits)} hits")
print(f"FDR < 0.05: {len(fdr_hits)} hits")
print(f"Druggable synthetic lethals (FDR < 0.05 & EffectSize < 0): {len(druggable_hits)} hits")

P-value < 0.05: 125104 hits
FDR < 0.05: 61 hits
Druggable synthetic lethals (FDR < 0.05 & EffectSize < 0): 25 hits
